# Ground-Truth Validation (최종본 v2) — GRNBoost2 / SpaGRN / SpaGTL

이전 버전(`groundtruth_validation_final.ipynb`, `db_val_script.py`) 대비 이번 노트북에서 바로잡은/재구성한 부분:

1. **Self-loop 제거 일관성 재점검 및 수정**
   - 예측 네트워크(REG)와 세 참조망(CollecTRI/Perturbation/ChIP-Atlas) 모두 로딩 시점에 self-loop(TF==target)를 명시적으로 제거합니다.
   - **실제로 발견된 버그**: GO/consensus 분석에 쓰이는 occurrence≥5 consensus edge 파일(`consensus_edges_occurrence_ge_5.tsv` 등)은 self-loop가 그대로 남아 있었습니다 (예: SpaGTL consensus 파일의 `Esrra → Esrra`). 참조망 쪽만 걸러지고 예측(특히 GO 입력) 쪽은 self-loop가 안 걸러진 지점이 실제로 있었다는 뜻이라, 이 노트북에서는 GO 입력 edge를 불러올 때도 self-loop를 다시 한번 명시적으로 제거합니다.
2. **run을 절대 합치지 않음**: GRNBoost2·SpaGTL의 10개 run 각각에서 독립적으로 AUPRC/EP/EPR/ER을 계산한 뒤에만 run 평균(±SD)을 냅니다. 한 번도 run들을 먼저 합쳐서(mean-importance로 병합) 계산하지 않습니다 — 이전 스크립트의 AUPRC 계산이 `MEAN_EDGES`(run 병합) 기준이었던 부분을 여기서는 run별 계산으로 교체했습니다.
3. **K sweep은 방법 자신의 predicted edge 수를 넘어서면 중단**: SpaGRN(~11만대), GRNBoost2(~2만대), SpaGTL(~5천~9천대)처럼 방법마다 전체 예측 규모가 다르므로, 그 run이 가진 predicted edge 수를 넘는 K는 계산하지 않고 빈 칸으로 남깁니다.
4. **메인 그림 2개**
   - **MAIN FIGURE 1**: 참조망별 EP@K / ER@K heatmap (Early Precision, Early Recall — EPR이 아님)
   - **MAIN FIGURE 2**: EP / EPR / AUPRC 통합 summary heatmap (K = 참조망 크기 고정, method × reference DB)
5. **Network 크기 vs 참조 데이터베이스 비교 figure를 맨 앞(Section 1)으로 이동**
6. **GO 분석**: GRNBoost2·SpaGTL은 occurrence≥5 consensus regulon(10 run 중 5회 이상 등장한 edge만) 사용, SpaGRN은 단일 run 그대로. Regulon당 평균 importance(=importance 합/target 수, target≥10) 기준 상위 **20개** regulon을 골라 target gene set 레벨로 GO:BP enrichment. 여러 regulon이 동일한 최상위 GO term에 매칭되면 그 중 p-value가 가장 작은 1개만 남기고 나머지는 제거.

## 지표 정의 (사용자 제공 `EarlyPrecision.py` / `AUPRC.py`를 이 프로젝트의 TF/target 스키마·multi-run 구조에 맞게 이식)

- **AUPRC**: universe = 참조망에 등장하는 유전자(TF∪target) 전체의 directed permutation(self-loop 제외). Label = 그 edge가 참조망에 있으면 1. Score = 예측 importance 절대값(없으면 0). `sklearn.precision_recall_curve` + `auc(recall, precision)`.
- **EP/EPR (fixed-K, BEELINE)**: K = |참조망 edge ∩ candidate universe|(고정), universe = 참조망 TF × 전체 gene universe(self-loop 제외). Tie-aware cutoff(“원본 EarlyPrecision.py”의 `bestVal = max(nonZeroMin, edgeWeightTopk)` 그대로 이식) 적용, 분모는 selected된 edge 수(고정 K가 아님 — tie 확장 때문). `EPR = EP / (K/|universe|)`.
- **ER (Early Recall)@K**: 선택된 edge 중 참조망 true edge를 맞춘 수 / 참조망 true edge 전체 수.
- 세 지표 모두 이 프로젝트의 실제 데이터 규모(참조망 크기가 축마다 다르고, 예측 방법마다 edge 수가 5천~11만으로 크게 다름)에 맞춰 universe/denominator를 계산합니다 — 즉 원본 BEELINE 스크립트의 `possible_edges`/분모 정의를 그대로 따르되, 이 프로젝트의 TF universe·gene universe·run 구조에 맞게 계산됩니다.

In [ ]:
from pathlib import Path
import glob, re, warnings
from itertools import permutations
import numpy as np, pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from sklearn.metrics import auc, precision_recall_curve
from kneed import KneeLocator

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
np.random.seed(0)

BASE = Path('/Users/kimchaeyoung/Desktop/thesis_data')
EXP_GRNBOOST2_GLOB = str(BASE / 'd0/multi_run/multi_run_analysis/expanded_post_pruning/run*_motif_pruned_edges_expanded.tsv')
EXP_SPAGTL_GLOB = str(BASE / 'spagtl/multi_run/multi_run_analysis/expanded_post_pruning/run*_motif_pruned_edges_expanded.tsv')
EXP_SPAGRN_PATH = BASE / 'spagrn/regulon_to_edge.csv'
ADATA_PATH = BASE / 'd0/sampled_adata.h5ad'
TF_LIST_PATH = BASE / 'allTFs_mm.txt'
COLLECTRI_PATH = BASE / 'external_data/collectri_mouse_genesymbol_clean.csv'
CHIPATLAS_CACHE = BASE / 'external_data/chipatlas_mm10_5kb_cache'
DEG_RAW_PATH = BASE / "perturb/differential expression of genes in all datasets(Mouse).txt"
OCC5_PATHS = {
    'GRNBoost2': BASE / 'd0/multi_run/multi_run_analysis/occurrence_consensus/motif_pruned_edges_occurrence_ge_05.tsv',
    'SpaGTL': BASE / 'spagtl/multi_run/cut_off_GO/consensus_edges/consensus_edges_occurrence_ge_5.tsv',
}

OUT = BASE / 'ground_validation_final_final2_outputs'
OUT.mkdir(exist_ok=True)
CPC_REF = BASE / 'cpc'  # 참고용(기존 Word 문서와 동일한 스타일의 레퍼런스 이미지) - 여기엔 쓰지 않음

METHOD_ORDER = ['GRNBoost2', 'SpaGRN', 'SpaGTL']
METHOD_COLORS = {'GRNBoost2': '#2a78d6', 'SpaGRN': '#008300', 'SpaGTL': '#eb6834'}
REF_LABEL = {'collectri': 'CollecTRI', 'perturbation': 'Perturbation', 'chipatlas': 'ChIP-Atlas'}
FILE_PREFIX = {'collectri': '1_collectri', 'perturbation': '2_perturbation', 'chipatlas': '3_chipatlas'}
TOPK_ABS_GRID = [500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]

def edge_set(df):
    return set(map(tuple, df[['TF', 'target']].itertuples(index=False, name=None)))


---
## 예측 네트워크(REG) + gene/TF universe 로딩 — self-loop은 로딩 시점에 제거

In [ ]:
def natural_run_key(p):
    m = re.search(r'run(\d+)', str(p), flags=re.I)
    return int(m.group(1)) if m else 10**9

def load_edges(path, method, run):
    d = pd.read_csv(path, sep='\t' if str(path).endswith(('.tsv', '.txt')) else ',')
    out = pd.DataFrame({
        'TF': d['TF'].astype(str).str.strip(),
        'target': d['target'].astype(str).str.strip(),
        'importance': pd.to_numeric(d['importance'], errors='coerce'),
    })
    out = out[out['TF'] != out['target']]  # self-loop 제거 (예측 네트워크)
    out = out.sort_values('importance', ascending=False, na_position='last').drop_duplicates(['TF', 'target'])
    out['method'] = method
    out['run'] = run
    return out.reset_index(drop=True)

def load_multirun(glob_pat, method):
    return {f'run{natural_run_key(f)}': load_edges(f, method, f'run{natural_run_key(f)}')
            for f in sorted(glob.glob(glob_pat), key=natural_run_key)}

print('예측 네트워크(REG) 로딩...')
REG = {
    'GRNBoost2': load_multirun(EXP_GRNBOOST2_GLOB, 'GRNBoost2'),
    'SpaGTL': load_multirun(EXP_SPAGTL_GLOB, 'SpaGTL'),
    'SpaGRN': {'single': load_edges(EXP_SPAGRN_PATH, 'SpaGRN', 'single')},
}

adata = ad.read_h5ad(ADATA_PATH, backed='r')
GENE_UNIV = set(adata.var_names.astype(str))
ALL_TFS = set(l.strip() for l in open(TF_LIST_PATH) if l.strip())
TF_UNIV = ALL_TFS & GENE_UNIV
TARGET_UNIV = GENE_UNIV
print(f'gene universe {len(GENE_UNIV):,} | TF universe {len(TF_UNIV):,}')

for m, runs in REG.items():
    ne = [len(v) for v in runs.values()]
    print(f'  {m}: {len(runs)} run(s), edge 수 {min(ne):,}~{max(ne):,} (run 평균 {np.mean(ne):,.0f} ± {np.std(ne):,.0f})')


---
## (선행 단계) ChIP-Atlas 원본 다운로드 — 로컬 캐시(`chipatlas_mm10_5kb_cache/`)에 없는 TF만 신규 다운로드

ChIP-Atlas의 target-genes API(`https://chip-atlas.dbcls.jp/data/{genome}/target/{antigen}.{dist_kb}.tsv`)는 antigen(TF) 기준으로 이미 유전자 단위 결합 테이블(`Target_genes`, `{TF}|Average`, 실험별 SRX 컬럼, STRING 컬럼)을 제공합니다. 여기서는 `Target_genes` -> `target`, `{TF}|Average` -> `avg_score` 두 컬럼만 뽑아 기존 캐시와 동일한 형식(`TF,target,avg_score`)으로 저장합니다.

- 예전에 쓰던 호스트(`dbarchive.biosciencedbc.jp`)는 현재 404이며, `chip-atlas.dbcls.jp`로 이전되었습니다.
- 이미 캐시된 TF는 재다운로드하지 않고 건너뜁니다(재실행해도 안전 — idempotent).
- ChIP-Atlas에 해당 antigen 실험이 아예 없는 TF는 404가 뜨며 자동으로 건너뜁니다(`not_available`).
- TF universe(1,487개) 전체를 시도하되, 실제로 캐시가 쌓이는 건 ChIP-Atlas가 커버하는 TF(현재 기준 341개)뿐입니다.

In [ ]:
# =========================================================
# ChIP-Atlas raw 다운로드 -- 캐시에 없는 TF만 새로 받아서 TF-gene(target, avg_score) 형식으로 저장
# =========================================================
import time
import requests
from io import StringIO

CHIPATLAS_GENOME = 'mm10'
CHIPATLAS_DIST_KB = 5
CHIPATLAS_URL_TMPL = 'https://chip-atlas.dbcls.jp/data/{genome}/target/{tf}.{dist_kb}.tsv'
RUN_CHIPATLAS_DOWNLOAD = True  # False로 두면 네트워크 요청 없이 기존 캐시만 사용

CHIPATLAS_CACHE.mkdir(parents=True, exist_ok=True)

def download_chipatlas_tf(tf, timeout=30):
    url = CHIPATLAS_URL_TMPL.format(genome=CHIPATLAS_GENOME, tf=tf, dist_kb=CHIPATLAS_DIST_KB)
    try:
        r = requests.get(url, timeout=timeout)
    except requests.RequestException as e:
        return 'failed', str(e)
    if r.status_code != 200 or not r.text.startswith('Target_genes'):
        return 'not_available', None
    df = pd.read_csv(StringIO(r.text), sep='\t')
    avg_col = f'{tf}|Average' if f'{tf}|Average' in df.columns else df.columns[1]
    out = df[['Target_genes', avg_col]].rename(columns={'Target_genes': 'target', avg_col: 'avg_score'})
    out.insert(0, 'TF', tf)
    out.to_csv(CHIPATLAS_CACHE / f'{tf}.csv', index=False)
    return 'downloaded', len(out)

if RUN_CHIPATLAS_DOWNLOAD:
    status_counts = {'cached': 0, 'downloaded': 0, 'not_available': 0, 'failed': 0}
    failed_tfs = []
    for tf in sorted(TF_UNIV):
        if (CHIPATLAS_CACHE / f'{tf}.csv').exists():
            status_counts['cached'] += 1
            continue
        status, info = download_chipatlas_tf(tf)
        status_counts[status] += 1
        if status == 'failed':
            failed_tfs.append((tf, info))
        time.sleep(0.3)  # ChIP-Atlas 서버 부담을 줄이기 위한 rate limit
    print('ChIP-Atlas 다운로드 결과:', status_counts)
    if failed_tfs:
        print(f'네트워크 실패 {len(failed_tfs)}건 (재실행하면 다시 시도됨):', failed_tfs[:10])
else:
    n_cached = len(list(CHIPATLAS_CACHE.glob('*.csv')))
    print(f'RUN_CHIPATLAS_DOWNLOAD=False -- 기존 캐시({n_cached}개 TF)만 사용')

---
## 참조망 3종 로딩(CollecTRI, Perturbation/KnockTF, ChIP-Atlas) — 세 참조망 모두 self-loop 명시적 제거

In [ ]:
print('참조망 로딩...')

collectri_raw = pd.read_csv(COLLECTRI_PATH)[['TF', 'target']].drop_duplicates()
COLLECTRI_REF_DF = collectri_raw[collectri_raw['TF'].isin(TF_UNIV) & collectri_raw['target'].isin(GENE_UNIV)
                                  & (collectri_raw['TF'] != collectri_raw['target'])].reset_index(drop=True)
print(f'CollecTRI: raw {len(collectri_raw):,} -> universe 필터 + self-loop 제거 후 {len(COLLECTRI_REF_DF):,} edges '
      f'(TF {COLLECTRI_REF_DF["TF"].nunique()}, target {COLLECTRI_REF_DF["target"].nunique()})')

usecols = ['Sample_ID', 'TF', 'Gene', 'Log2FC', 'Corrected_P', 'up_down']
chunks_raw, chunks_f = [], []
for ch in pd.read_csv(DEG_RAW_PATH, sep='\t', usecols=usecols, chunksize=500000, low_memory=False):
    ch['Corrected_P'] = pd.to_numeric(ch['Corrected_P'], errors='coerce')
    ch = ch[(ch['up_down'] == 2) & (ch['Corrected_P'] < 0.05)]
    chunks_raw.append(ch[['TF', 'Gene']])
    chunks_f.append(ch[ch['TF'].isin(TF_UNIV) & ch['Gene'].isin(GENE_UNIV) & (ch['TF'] != ch['Gene'])][['TF', 'Gene']])
perturb_raw = pd.concat(chunks_raw, ignore_index=True).drop_duplicates().rename(columns={'Gene': 'target'})
PERTURB_REF_DF = pd.concat(chunks_f, ignore_index=True).drop_duplicates().rename(columns={'Gene': 'target'})
print(f'Perturbation: raw {len(perturb_raw):,} -> universe 필터 + self-loop 제거 후 {len(PERTURB_REF_DF):,} edges, '
      f'{PERTURB_REF_DF["TF"].nunique()} TFs')

def elbow_cutoff(y):
    y = np.asarray(y, dtype=float); x = np.arange(len(y))
    if len(y) < 10 or y.max() == y.min():
        return max(1, int(len(y) * 0.1))
    try:
        kl = KneeLocator(x, y, curve='convex', direction='decreasing', S=1.0)
        if kl.knee is not None and kl.knee > 0:
            return int(kl.knee)
    except Exception:
        pass
    return max(1, int(len(y) * 0.1))

cache_files = sorted(CHIPATLAS_CACHE.glob('*.csv'))
chip_tables = {f.stem: pd.read_csv(f) for f in cache_files}
chip_raw = pd.concat([d[['TF', 'target']].assign(TF=tf) for tf, d in chip_tables.items()],
                      ignore_index=True).drop_duplicates(['TF', 'target'])
chip_ref_rows = []
for tf, d in chip_tables.items():
    if tf not in TF_UNIV:
        continue
    d = d[d['target'].isin(GENE_UNIV) & (d['target'] != tf)].sort_values('avg_score', ascending=False).reset_index(drop=True)
    if len(d) < 10:
        continue
    k = elbow_cutoff(d['avg_score'].to_numpy())
    top = d.head(k)
    top = top[top['avg_score'] > 0]
    chip_ref_rows.append(top[['TF', 'target']].assign(TF=tf))
CHIPATLAS_REF_DF = pd.concat(chip_ref_rows, ignore_index=True).drop_duplicates(['TF', 'target'])
print(f'ChIP-Atlas: raw {len(chip_raw):,} -> universe 필터 + elbow cutoff + self-loop 제거 후 {len(CHIPATLAS_REF_DF):,} edges, '
      f'{CHIPATLAS_REF_DF["TF"].nunique()} TFs')

REFS = {'collectri': COLLECTRI_REF_DF, 'perturbation': PERTURB_REF_DF, 'chipatlas': CHIPATLAS_REF_DF}
REF_SETS = {k: edge_set(v) for k, v in REFS.items()}


---
# MAIN FIGURE 0 — Section 1. Network 크기 비교: 예측 방법 vs 참조 데이터베이스

정량 검증(EP/EPR/AUPRC)에 들어가기 전에, 비교 대상이 되는 예측 네트워크와 참조 데이터베이스의 **기본 규모(edge/gene/TF 수)**가 서로 어느 정도 스케일인지 가장 먼저 확인합니다 — 이후 모든 top-K 기반 비교를 해석하는 전제 조건이기 때문에 맨 앞에 배치합니다.

- 예측 방법(GRNBoost2·SpaGTL)은 **10개 run 각각에서 독립적으로 edge/gene/TF 수를 센 뒤 run 평균 ± SD**로 집계합니다(합쳐서 union으로 세지 않음). SpaGRN은 결정론적 단일 run.
- 참조 DB는 raw(mouse) 규모와, 우리 gene/TF universe로 필터링 + self-loop 제거 후 규모를 함께 표시합니다.


In [ ]:
def network_stats_per_run(method):
    rows = []
    for run, d in REG[method].items():
        tf_set = set(d['TF'])
        n_nodes = len(tf_set | set(d['target']))
        rows.append({'method': method, 'run': run, 'n_edges': len(d), 'n_nodes': n_nodes,
                     'n_TF': d['TF'].nunique(), 'n_target': d['target'].nunique()})
    return pd.DataFrame(rows)

stats_df = pd.concat([network_stats_per_run(m) for m in METHOD_ORDER], ignore_index=True)
stats_summary = stats_df.groupby('method')[['n_edges', 'n_nodes', 'n_TF', 'n_target']].agg(['mean', 'std']).reindex(METHOD_ORDER)
stats_summary.to_csv(OUT / 'network_stats_edge_node_tf_target.csv')
print('=== 예측 방법 네트워크 규모 (run 평균 ± SD) ===')
print(stats_summary.round(1))

def edge_node_tf_counts(df):
    tf_set = set(df['TF'])
    return dict(n_edges=len(df), n_nodes=len(tf_set | set(df['target'])), n_TF=df['TF'].nunique())

db_size_rows = []
for name, raw_df, filt_df in [('CollecTRI', collectri_raw, COLLECTRI_REF_DF),
                               ('Perturbation', perturb_raw, PERTURB_REF_DF),
                               ('ChIP-Atlas', chip_raw, CHIPATLAS_REF_DF)]:
    c_raw = edge_node_tf_counts(raw_df); c_filt = edge_node_tf_counts(filt_df)
    db_size_rows.append({'source': name, 'stage': 'raw(mouse)', 'n_edges_sd': 0, 'n_nodes_sd': 0, 'n_TF_sd': 0, **c_raw})
    db_size_rows.append({'source': name, 'stage': 'gene_universe_filtered', 'n_edges_sd': 0, 'n_nodes_sd': 0, 'n_TF_sd': 0, **c_filt})
for m in METHOD_ORDER:
    db_size_rows.append({'source': m, 'stage': 'predicted(run mean)',
                          'n_edges': stats_summary.loc[m, ('n_edges', 'mean')], 'n_edges_sd': stats_summary.loc[m, ('n_edges', 'std')] or 0,
                          'n_nodes': stats_summary.loc[m, ('n_nodes', 'mean')], 'n_nodes_sd': stats_summary.loc[m, ('n_nodes', 'std')] or 0,
                          'n_TF': stats_summary.loc[m, ('n_TF', 'mean')], 'n_TF_sd': stats_summary.loc[m, ('n_TF', 'std')] or 0})
db_size_df = pd.DataFrame(db_size_rows)
db_size_df.to_csv(OUT / 'network_and_db_size_comparison.csv', index=False)
print('\n=== 예측 방법(run 평균±SD) + 참조 DB 크기 비교 (raw vs universe-filtered) ===')
print(db_size_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5.6))
plot_rows = db_size_df[db_size_df['stage'] != 'raw(mouse)'].copy()
labels = plot_rows['source'] + '\n(' + plot_rows['stage'].str.replace('gene_universe_filtered', 'filtered').str.replace('predicted(run mean)', 'predicted') + ')'
bar_colors = [METHOD_COLORS.get(s, '#898781') for s in plot_rows['source']]
x = np.arange(len(plot_rows))
for ax, (col, title) in zip(axes, [('n_edges', 'Total Number of Edges'), ('n_nodes', 'Total Number of Genes'), ('n_TF', 'Total Number of TFs')]):
    ax.bar(x, plot_rows[col], yerr=plot_rows[col + '_sd'], capsize=4, color=bar_colors)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=10)
    ax.set_title(title, fontsize=14); ax.set_ylabel('Count', fontsize=12)
    ax.tick_params(axis='y', labelsize=10)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Predicted Methods (run mean ± SD) vs Reference Databases: Network Size Comparison', fontsize=16)
plt.tight_layout()
fig.savefig(OUT / 'fig_network_size_comparison.png', bbox_inches='tight', dpi=200)
plt.show()


---
## Section 2. Universe / Baseline 계산 (참조망별로 다름)

AUPRC의 universe(및 baseline=prevalence)는 **참조망에 등장하는 유전자(TF∪target) 전체**를 기준으로,
EPR의 universe는 **참조망 TF × 전체 측정 유전자(gene universe)** 를 기준으로 하므로 서로 다르게 계산됩니다.
두 universe 정의 모두 self-loop을 제외합니다(참조망 로딩 단계에서 이미 제거되었지만, 아래서 한 번 더 방어적으로 필터링합니다).

In [ ]:
def edge_universe_auprc(ref_df):
    genes = sorted(set(ref_df['TF']).union(set(ref_df['target'])))
    n_possible = len(genes) * (len(genes) - 1)
    n_true = len(set(zip(ref_df['TF'], ref_df['target'])))
    return n_possible, n_true, n_true / n_possible

def beeline_prep_universe(ref_df):
    """EPR/EP/ER의 universe(=possible_edges) 및 참조망 true_edges를 1회만 준비.
    BEELINE의 EarlyPrecision.py _build_edge_universe(tf_edges=True)와 동일하게,
    possible_edges = 참조망 TF x 참조망에 실제 등장하는 유전자(unique_nodes), self-loop 제외.
    (기존에는 target 쪽을 GENE_UNIV(데이터셋 전체 측정 유전자, 16,051개)로 확장해서 썼으나,
    BEELINE 원본 정의와 다르고 그만큼 baseline이 작아져 EPR이 과대평가됐음 -- 원본 정의로 수정.)"""
    ref_no_self = ref_df[ref_df['TF'] != ref_df['target']].drop_duplicates()
    ref_tfs = set(ref_no_self['TF'])
    tgt_univ_set = set(ref_df['TF']) | set(ref_df['target'])
    n_possible = len(ref_tfs) * len(tgt_univ_set) - len(ref_tfs & tgt_univ_set)
    true_edges = {(tf, tg) for tf, tg in zip(ref_no_self['TF'], ref_no_self['target']) if tg in tgt_univ_set}
    return dict(ref_tfs=ref_tfs, tgt_univ_set=tgt_univ_set, n_possible=n_possible, true_edges=true_edges)

print(f"{'참조망':<15}{'AUPRC |universe|':>18}{'AUPRC baseline':>16}{'EPR |universe|':>16}{'EPR random_prec':>18}")
AUPRC_BASELINE = {}
EPR_UNIVERSE = {}
for name, ref_df in REFS.items():
    n_poss, n_true, base = edge_universe_auprc(ref_df)
    AUPRC_BASELINE[name] = base
    universe = beeline_prep_universe(ref_df)
    EPR_UNIVERSE[name] = universe
    rp = len(universe['true_edges']) / universe['n_possible']
    print(f"{REF_LABEL[name]:<15}{n_poss:>18,}{base:>16.6f}{universe['n_possible']:>16,}{rp:>18.6f}")

---
## Section 3. 평가 함수 (BEELINE 표준 정의를 이 프로젝트 구조에 맞게 이식)

`compute_auprc_beeline`과 `beeline_ep_epr`은 사용자 제공 `AUPRC.py` / `EarlyPrecision.py`의 계산 로직(고정 universe, tie-aware cutoff, selected-count 분모)을 그대로 재현하되, 참조 스크립트의 범용 `Evaluator`/`EvaluationData` 클래스 구조 대신 이 프로젝트의 `TF/target/importance` 컬럼 + multi-run 딕셔너리 구조에 맞춰 이식했습니다 (요청하신 "우리한테 맞출 denominator/구조가 있으면 그렇게" 반영).

In [ ]:
def compute_auprc_beeline(pred_df, ref_df):
    """AUPRC.py의 _compute_auprc 이식. universe=참조망 유전자 전체의 permutation(self-loop 제외)."""
    gt_genes = sorted(set(ref_df['TF']).union(set(ref_df['target'])))
    possible_edges = list(permutations(gt_genes, 2))
    true_edge_set = set(zip(ref_df['TF'], ref_df['target']))
    pred = pred_df[pred_df['TF'] != pred_df['target']].copy()
    pred['_abs'] = pred['importance'].abs()
    pred = pred.sort_values('_abs', ascending=False).drop_duplicates(subset=['TF', 'target'])
    pred_lookup = dict(zip(zip(pred['TF'], pred['target']), pred['_abs'].astype(float)))
    true_labels = [1 if e in true_edge_set else 0 for e in possible_edges]
    pred_scores = [pred_lookup.get(e, 0.0) for e in possible_edges]
    if sum(true_labels) == 0:
        return float('nan')
    precision, recall, _ = precision_recall_curve(true_labels, pred_scores)
    return float(auc(recall, precision))

def beeline_ep_epr(pred_df, universe, k_override=None):
    """EarlyPrecision.py의 _compute_early_precision 이식(tf_edges=True 모드) + ER(Early Recall) 추가.
    K 기본값 = |true_edges|(참조망 크기). k_override로 K sweep 지원."""
    ref_tfs = universe['ref_tfs']; tgt_univ_set = universe['tgt_univ_set']
    n_possible = universe['n_possible']; true_edges = universe['true_edges']
    K = k_override if k_override is not None else len(true_edges)
    baseline = K / n_possible if n_possible else np.nan
    if K == 0 or n_possible == 0 or not np.isfinite(baseline) or baseline <= 0:
        return dict(k_selected=0, EP=np.nan, EPR=np.nan, ER=np.nan, baseline=baseline)

    e = pred_df[(pred_df['TF'] != pred_df['target']) & pred_df['TF'].isin(ref_tfs) & pred_df['target'].isin(tgt_univ_set)].copy()
    e['_abs'] = e['importance'].abs()
    e = e.sort_values('_abs', ascending=False).drop_duplicates(['TF', 'target']).reset_index(drop=True)
    if len(e) == 0:
        return dict(k_selected=0, EP=0.0, EPR=0.0, ER=0.0, baseline=baseline)

    maxk = min(len(e), K)
    edge_weight_topk = float(e.iloc[maxk - 1]['_abs'])
    nonzero = e.loc[e['_abs'] > 0, '_abs']
    non_zero_min = float(nonzero.min()) if len(nonzero) else 0.0
    best_val = max(non_zero_min, edge_weight_topk)  # tie-aware cutoff (원본 EarlyPrecision.py 그대로)
    selected = e[e['_abs'] >= best_val]
    if len(selected) == 0:
        return dict(k_selected=0, EP=0.0, EPR=0.0, ER=0.0, baseline=baseline)

    selected_edges = set(zip(selected['TF'], selected['target']))
    hits = len(selected_edges & true_edges)
    EP = hits / len(selected_edges)
    EPR = EP / baseline if baseline > 0 else np.nan
    ER = hits / len(true_edges) if len(true_edges) else np.nan
    return dict(k_selected=int(len(selected_edges)), EP=float(EP), EPR=float(EPR), ER=float(ER), baseline=float(baseline))

def topk_ep_er_by_K(pred_df, universe, tf_univ, tgt_univ, k_grid=TOPK_ABS_GRID):
    """K를 키워가며 EP/EPR/ER 계산. K가 '이 run 자신의 predicted edge 전체 개수'(tf/gene universe로만
    필터링, ref_tfs로는 필터링하지 않은 전체 규모)를 넘어서면 그 지점부터 NaN — 존재하지 않는 edge를
    억지로 채우지 않기 위함(= 사용자가 말한 "예측 edge가 바닥나면 중단")."""
    n_total = len(pred_df[pred_df['TF'].isin(tf_univ) & pred_df['target'].isin(tgt_univ)])
    rows = []
    for k in k_grid:
        if k > n_total:
            rows.append({'k': k, 'EP': np.nan, 'EPR': np.nan, 'ER': np.nan})
            continue
        stat = beeline_ep_epr(pred_df, universe, k_override=k)
        rows.append({'k': k, 'EP': stat['EP'], 'EPR': stat['EPR'], 'ER': stat['ER']})
    return pd.DataFrame(rows)

print('평가 함수 정의 완료')


---
## Section 4. 전체 계산 실행 — **run을 절대 합치지 않고**, run별로 독립 계산한 뒤에만 평균(±SD)

- `fixed_perrun_df`: K = 참조망 크기(고정, BEELINE 대표값) 기준 AUPRC / EP / EPR / ER — MAIN FIGURE 2의 재료
- `byk_perrun_df`: K = 500~100,000 sweep, run 자신의 predicted edge 수를 넘으면 중단 — MAIN FIGURE 1의 재료


In [ ]:
fixed_rows = []
byk_rows = []
for ref_name, ref_df in REFS.items():
    universe = EPR_UNIVERSE[ref_name]
    for m in METHOD_ORDER:
        for run, pred in REG[m].items():
            auprc = compute_auprc_beeline(pred, ref_df)
            stat = beeline_ep_epr(pred, universe)  # K=|true_edges| 고정
            fixed_rows.append({'ref': ref_name, 'method': m, 'run': run, 'AUPRC': auprc,
                                'EP': stat['EP'], 'EPR': stat['EPR'], 'ER': stat['ER'], 'k_selected': stat['k_selected']})

            byk = topk_ep_er_by_K(pred, universe, TF_UNIV, TARGET_UNIV, TOPK_ABS_GRID)
            byk['ref'] = ref_name; byk['method'] = m; byk['run'] = run
            byk_rows.append(byk)
    print(f'{REF_LABEL[ref_name]} 계산 완료 (run별 독립 계산)')

fixed_perrun_df = pd.DataFrame(fixed_rows)
fixed_perrun_df.to_csv(OUT / 'fixed_K_perrun_AUPRC_EP_EPR_ER.csv', index=False)
byk_perrun_df = pd.concat(byk_rows, ignore_index=True)
byk_perrun_df.to_csv(OUT / 'byK_perrun_EP_EPR_ER.csv', index=False)

print('\n=== run 평균 ± SD (K=참조망 크기 고정) ===')
print(fixed_perrun_df.groupby(['ref', 'method'])[['AUPRC', 'EP', 'EPR', 'ER']].agg(['mean', 'std']).round(4))


---
# MAIN FIGURE 1 — EP@K & ER@K Heatmap (참조망별, run 평균)

고정 K(참조망 크기) 대신 K=500~100,000으로 계속 키워가며 Early Precision / Early Recall을 봅니다.
**빈 칸 = 그 방법(run)의 predicted edge pool이 그 K에 도달하기 전에 바닥남**(SpaGTL은 보통 5,000~9,000대에서, GRNBoost2는 20,000대에서, SpaGRN은 100,000대까지 채워짐).

In [ ]:
def plot_main_ep_er_heatmap(ref_name):
    sub = byk_perrun_df[byk_perrun_df['ref'] == ref_name]
    byk_mean = sub.groupby(['method', 'k'])[['EP', 'EPR', 'ER']].mean().reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(14.4, 5.2))
    panels = [('EP', 'Early Precision (EP)@K', 'viridis', '{:.4f}'),
              ('ER', 'Early Recall (ER)@K', 'viridis', '{:.4f}')]
    for ax, (col, title, cmap, fmt) in zip(axes, panels):
        piv = byk_mean.pivot_table(index='method', columns='k', values=col, sort=False).reindex(METHOD_ORDER)[TOPK_ABS_GRID]
        data = piv.values.astype(float)
        masked = np.ma.masked_invalid(data)
        im = ax.imshow(masked, cmap=cmap, aspect='auto')
        ax.set_xticks(range(len(TOPK_ABS_GRID)))
        ax.set_xticklabels([f'{k:,}' for k in TOPK_ABS_GRID], fontsize=12, rotation=40, ha='right')
        ax.set_yticks(range(len(METHOD_ORDER))); ax.set_yticklabels(METHOD_ORDER, fontsize=14)
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                v = data[i, j]
                if np.isfinite(v):
                    color = 'white' if (v < np.nanmean(data)) else 'black'
                    ax.text(j, i, fmt.format(v), ha='center', va='center', fontsize=12, color=color)
        cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        cb.ax.tick_params(labelsize=12)
        ax.set_title(title, fontsize=19, pad=10)
        ax.set_xlabel('Top-K predicted edges', fontsize=14)
    fig.suptitle(f'{REF_LABEL[ref_name]}: Early Precision & Early Recall @ Top-K (MAIN, run mean)\n'
                 f'blank cell = predicted-edge pool exhausted before reaching that K', fontsize=20, y=1.06)
    plt.tight_layout()
    prefix = FILE_PREFIX[ref_name]
    fig.savefig(OUT / f'{prefix}_MAIN_ep_er_by_K_heatmap.png', bbox_inches='tight', dpi=200)
    plt.show()

for ref_name in REFS:
    plot_main_ep_er_heatmap(ref_name)


---
# MAIN FIGURE 2 — EP / EPR / AUPRC Summary Heatmap (K = 참조망 크기, method × reference DB)

BEELINE 표준 대표값(K = 참조망 edge 수 그 자체, tie-aware cutoff)에서의 Early Precision, Early Precision Ratio, AUPRC를 세 참조망에 대해 한 번에 비교하는 메인 요약 그림입니다. 모든 값은 run별로 독립 계산한 뒤 run 평균입니다(SpaGRN은 단일 run).

In [ ]:
agg = fixed_perrun_df.groupby(['method', 'ref'])[['EP', 'EPR', 'AUPRC']].mean().reset_index()
refs_order = list(REF_LABEL.keys())
ep_summary = agg.pivot_table(index='method', columns='ref', values='EP').reindex(METHOD_ORDER)[refs_order]
ep_summary.columns = [REF_LABEL[c] for c in ep_summary.columns]
epr_summary = agg.pivot_table(index='method', columns='ref', values='EPR').reindex(METHOD_ORDER)[refs_order]
epr_summary.columns = [REF_LABEL[c] for c in epr_summary.columns]
auprc_summary = agg.pivot_table(index='method', columns='ref', values='AUPRC').reindex(METHOD_ORDER)[refs_order]
auprc_summary.columns = [REF_LABEL[c] for c in auprc_summary.columns]

ep_summary.to_csv(OUT / 'MAIN_ep_summary_table.csv')
epr_summary.to_csv(OUT / 'MAIN_epr_summary_table.csv')
auprc_summary.to_csv(OUT / 'MAIN_auprc_summary_table.csv')
print('=== EP (method x ref, run 평균) ==='); print(ep_summary.round(4))
print('=== EPR (method x ref, run 평균) ==='); print(epr_summary.round(3))
print('=== AUPRC (method x ref, run 평균) ==='); print(auprc_summary.round(4))
print('=== AUPRC random baseline(prevalence, method 무관) ===')
print({REF_LABEL[k]: round(v, 6) for k, v in AUPRC_BASELINE.items()})

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, (df, title, fmt) in zip(axes, [
    (ep_summary, 'Early Precision (EP)', '{:.4f}'),
    (epr_summary, 'EPR (Early Precision Ratio)', '{:.2f}'),
    (auprc_summary, 'AUPRC', '{:.4f}'),
]):
    data = df.values.astype(float)
    im = ax.imshow(data, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(df.columns))); ax.set_xticklabels(df.columns, fontsize=12)
    ax.set_yticks(range(len(df.index))); ax.set_yticklabels(df.index, fontsize=13)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            if np.isfinite(v):
                ax.text(j, i, fmt.format(v), ha='center', va='center', fontsize=12,
                         color='white' if v < np.nanmean(data) else 'black')
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=11)
    ax.set_title(title, fontsize=16, pad=8)

# AUPRC 패널에 참조망별 random baseline(prevalence)을 subtitle로 표기 — 절대값인 AUPRC는
# baseline 없이는 "높다/낮다"를 판단할 수 없음(EPR은 정의상 baseline=1이 내장돼 있어 불필요).
auprc_ax = axes[2]
baseline_str = ' | '.join(f'{REF_LABEL[k]}={AUPRC_BASELINE[k]:.4f}' for k in refs_order)
auprc_ax.set_xlabel(f'random baseline (prevalence): {baseline_str}', fontsize=9.5)

fig.suptitle('MAIN: EP / EPR / AUPRC Summary (BEELINE, K = reference network size, run mean): Method x Reference Database',
             fontsize=17, y=1.05)
plt.tight_layout()
fig.savefig(OUT / 'MAIN_ep_epr_auprc_summary_heatmap.png', bbox_inches='tight', dpi=200)
plt.show()


---
# GO Enrichment — Consensus Regulon Top-20 (평균 importance 기준)

- **GRNBoost2·SpaGTL**: occurrence≥5 consensus edge(10개 run 중 5회 이상 등장한 edge)만 사용 — 단발성 run 노이즈 제거.
- **SpaGRN**: 단일 run 그대로 사용(애초에 run이 1개라 consensus 개념이 적용되지 않음).
- **self-loop 재확인**: consensus edge 파일에 self-loop이 남아있는 경우가 실제로 있었음(예: SpaGTL의 `Esrra → Esrra`) — GO 입력으로 쓰기 전에 여기서 다시 한번 명시적으로 제거합니다.
- regulon 랭킹 기준 = **target당 평균 importance**(= importance 합 / target 수), target 수 10개 미만인 TF는 통계적으로 불안정하므로 제외.
- 방법별 **top-20** regulon의 target gene set으로 g:Profiler GO:BP enrichment(background=gene universe, domain_scope=custom).
- **중복 제거**: 같은 방법 안에서 서로 다른 regulon(TF)이 우연히 동일한 최상위 GO term에 매칭되면, p-value가 가장 작은 1개만 남기고 나머지는 제거합니다.

In [ ]:
def load_go_input_edges(method):
    if method == 'SpaGRN':
        d = REG['SpaGRN']['single'][['TF', 'target', 'importance']].copy()
    else:
        raw = pd.read_csv(OCC5_PATHS[method], sep='\t')
        d = raw[['TF', 'target', 'importance']].copy()
        d['TF'] = d['TF'].astype(str).str.strip()
        d['target'] = d['target'].astype(str).str.strip()
    n_before = len(d)
    d = d[d['TF'] != d['target']].reset_index(drop=True)  # self-loop 제거 (consensus 파일에 실제로 존재하는 것을 확인)
    n_removed = n_before - len(d)
    if n_removed:
        print(f'  [self-loop 제거] {method}: consensus/regulon edge 중 {n_removed}개 self-loop 제거')
    return d

MIN_TARGETS_FOR_AVG_IMPORTANCE = 10
GO_TOPN = 20

def topN_regulons_by_avg_importance(d, n=GO_TOPN, min_targets=MIN_TARGETS_FOR_AVG_IMPORTANCE):
    g = d.groupby('TF').agg(imp_sum=('importance', 'sum'), n_target=('target', 'nunique'))
    g['avg_importance'] = g['imp_sum'] / g['n_target']
    g = g[g['n_target'] >= min_targets]
    return g.sort_values('avg_importance', ascending=False).head(n)

from gprofiler import GProfiler
gpr = GProfiler(return_dataframe=True)
GENE_UNIV_LIST = sorted(GENE_UNIV)
GPROFILER_TOPN_TARGETS = 100

bubble_rows = []
top_tf_rows = []
for m in METHOD_ORDER:
    edges = load_go_input_edges(m)
    top_tf_df = topN_regulons_by_avg_importance(edges, n=GO_TOPN)
    print(f'{m}: top-{GO_TOPN}(평균 importance 기준, consensus/self-loop 제외) TF = {list(top_tf_df.index)}')
    for tf, row in top_tf_df.iterrows():
        top_tf_rows.append({'method': m, 'TF': tf, 'avg_importance': row['avg_importance'], 'n_target': row['n_target']})
        tf_targets = edges[edges['TF'] == tf].sort_values('importance', ascending=False)
        query_genes = sorted(set(tf_targets['target'].head(GPROFILER_TOPN_TARGETS)) & GENE_UNIV)
        if len(query_genes) < 5:
            continue
        try:
            res = gpr.profile(organism='mmusculus', query=query_genes, background=GENE_UNIV_LIST,
                               domain_scope='custom', sources=['GO:BP'],
                               significance_threshold_method='fdr', user_threshold=0.05)
        except Exception as e:
            print(f'  [error] {m}/{tf}: {e}')
            continue
        if res is None or len(res) == 0:
            continue
        top_term = res.sort_values('p_value').iloc[0]
        bubble_rows.append({'method': m, 'TF': tf, 'name': top_term['name'], 'p_value': float(top_term['p_value']),
                             'intersection_size': top_term.get('intersection_size', np.nan)})

top_tf_df_all = pd.DataFrame(top_tf_rows)
top_tf_df_all.to_csv(OUT / 'go_top20_regulon_TFs_table.csv', index=False)
bubble_df = pd.DataFrame(bubble_rows)
bubble_df.to_csv(OUT / 'go_bubble_top20_avgimportance_raw_table.csv', index=False)

# 중복 GO term 제거: 같은 method 안에서 동일한 term에 여러 TF가 매칭되면 p-value 최소(가장 유의) 1개만 남김
bubble_df_dedup = (bubble_df.sort_values('p_value')
                    .groupby(['method', 'name'], as_index=False).first())
bubble_df_dedup.to_csv(OUT / 'go_bubble_top20_avgimportance_dedup_table.csv', index=False)
print(f'\nGO bubble: 원본 {len(bubble_df)}개 (TF,term) 쌍 -> 중복 term 제거 후 {len(bubble_df_dedup)}개')

In [ ]:
if len(bubble_df_dedup):
    max_rows = bubble_df_dedup.groupby('method').size().max()
    fig, axes = plt.subplots(1, 3, figsize=(32, max(8, max_rows * 0.6)), sharey=False)
    for ax, m in zip(axes, METHOD_ORDER):
        sub = bubble_df_dedup[bubble_df_dedup['method'] == m].copy()
        if len(sub) == 0:
            ax.set_title(f'{m}\n(no GO result)', fontsize=16)
            continue
        sub['label'] = sub['TF'] + ' — ' + sub['name']
        sub['neglog10p'] = -np.log10(sub['p_value'].clip(lower=1e-300))
        sub = sub.sort_values('p_value', ascending=False).reset_index(drop=True)
        y = np.arange(len(sub))
        sc = ax.scatter(sub['neglog10p'], y, s=sub['intersection_size'] * 6 + 40,
                         c=sub['neglog10p'], cmap='viridis', edgecolor='k', linewidth=0.4, alpha=0.85)
        ax.set_yticks(y); ax.set_yticklabels(sub['label'], fontsize=13)
        ax.set_xlabel('-log10(p-value)', fontsize=14)
        ax.tick_params(axis='x', labelsize=12)
        ax.set_title(f'{m} — Top-{GO_TOPN} Regulon (avg importance) GO:BP', fontsize=16)
        ax.grid(axis='x', color='#e1e0d9'); ax.set_axisbelow(True)
        cb = plt.colorbar(sc, ax=ax, fraction=0.02, pad=0.02)
        cb.ax.tick_params(labelsize=11)
        cb.set_label('-log10(p)', fontsize=13)
    fig.suptitle(f'Top-{GO_TOPN} Regulon (target당 평균 importance 기준, consensus occurrence≥5) — GO:BP Bubble Plot\n'
                 '(bubble 크기 = overlap gene 수, y축 = TF — 가장 유의한 매칭 기능 1개, 중복 term은 최상위 1개만 표시)', fontsize=18)
    plt.tight_layout()
    fig.savefig(OUT / 'MAIN_go_bubble_top20_avgimportance.png', bbox_inches='tight', dpi=200)
    plt.show()
else:
    print('GO 결과가 없어 bubble plot을 생성하지 못했습니다.')

---
# 최종 요약

- **Section 1 (맨 앞)**: 예측 방법(run 평균±SD) vs 참조 DB의 edge/gene/TF 규모를 가장 먼저 확인 — 이후 모든 top-K 비교의 전제.
- **MAIN FIGURE 1**: 참조망별 EP@K / ER@K heatmap — 방법 자신의 predicted edge pool이 소진되는 지점(SpaGTL ~5천~9천, GRNBoost2 ~2만, SpaGRN ~10만)에서 자동으로 멈춤.
- **MAIN FIGURE 2**: EP / EPR / AUPRC 통합 summary heatmap (K = 참조망 크기 고정, method × reference DB) — 모두 run별 독립 계산 후 run 평균.
- **GO**: GRNBoost2·SpaGTL은 occurrence≥5 consensus regulon, SpaGRN은 단일 run, 평균 importance 기준 top-20 regulon, self-loop 제외, 중복 GO term은 최상위 1개만 표시.
- 이번 버전에서 실제로 고친 것: (1) GO consensus edge 입력에 남아있던 self-loop 제거, (2) AUPRC를 run 병합(MEAN_EDGES) 대신 run별 독립 계산 후 평균으로 교체, (3) Network 크기 비교 figure를 맨 앞으로 재배치, (4) GO top-20 regulon + 중복 term 제거 로직 적용.